In [0]:
# lab_backlog_raw_ingest
import xlrd
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException

In [0]:
VOLUME = "/Volumes/opsanalytics_adb_workspace01/lab/raw_data/backlog_data"
RAW    = "opsanalytics_adb_workspace01.lab.cyto_backlog_raw"
PREFIX = "KPI REPORT - CYTOLOGY PENDING CASES"

In [0]:
# --- files not yet ingested -------------------------------------------------
all_files = [f.path for f in dbutils.fs.ls(VOLUME)
             if f.name.lower().endswith((".xls", ".xlsx"))
             and f.name.startswith(PREFIX)]

try:
    done = {r._source_file for r in spark.table(RAW).select("_source_file").distinct().collect()}
except AnalysisException:
    done = set()

pending = sorted(f for f in all_files if f.split("/")[-1] not in done)
print(f"{len(pending)} file(s) to ingest")

# --- land each file as-is ---------------------------------------------------
for path in pending:
    fname = path.split("/")[-1]
    local = path.replace("dbfs:/Volumes", "/Volumes")

    m = re.search(r"(\d{4}-\d{2}-\d{2})", fname)
    if not m:
        print(f"SKIP (no date in filename): {fname}")
        continue
    report_date = m.group(1)

    pdf = pd.read_excel(local, sheet_name=0, skiprows=1, dtype=str)
    pdf = pdf.iloc[:-1]
    pdf.columns = [str(c).strip().replace(" ", "_").replace(".", "_") for c in pdf.columns]
    pdf = pdf.loc[:, [c for c in pdf.columns if not c.startswith("Unnamed")]]

    if pdf.empty:
        print(f"SKIP (empty): {fname}")
        continue

    sdf = (spark.createDataFrame(pdf)
           .withColumn("_source_file", F.lit(fname))
           .withColumn("_report_date", F.to_date(F.lit(report_date)))
           .withColumn("_ingested_at", F.current_timestamp()))

    (sdf.write.mode("append").option("mergeSchema", "true").saveAsTable(RAW))
    print(f"OK ({sdf.count()} rows): {fname}")